## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Start with imports
import os
import json
import google.auth
import google.auth.transport.requests
from dotenv import load_dotenv
from openai import OpenAI          # used for Vertex AI OpenAI-compatible interface
from IPython.display import Markdown, display


In [ ]:
# Load env and initialise Vertex AI credentials
load_dotenv(override=True)

project_id = os.getenv("GCP_PROJECT")
location   = os.getenv("GCP_LOCATION", "us-central1")

# Refresh Application Default Credentials
credentials, _ = google.auth.default()
auth_req = google.auth.transport.requests.Request()
credentials.refresh(auth_req)

def vertex_client():
    """Return a fresh Vertex AI OpenAI-compatible client (token valid ~1 h)."""
    creds, _ = google.auth.default()
    req = google.auth.transport.requests.Request()
    creds.refresh(req)
    return OpenAI(
        base_url=(
            f"https://{location}-aiplatform.googleapis.com/v1/projects/{project_id}"
            f"/locations/{location}/endpoints/openapi"
        ),
        api_key=creds.token,
    )

client = vertex_client()
print(f"Vertex AI client ready  —  project={project_id}  location={location}")


In [ ]:
# Verify GCP credentials are available
try:
    creds_check, proj = google.auth.default()
    req = google.auth.transport.requests.Request()
    creds_check.refresh(req)
    print(f"GCP credentials OK  — project: {proj or project_id}")
except google.auth.exceptions.DefaultCredentialsError as e:
    print(f"Credentials not found: {e}")
    print("Run:  gcloud auth application-default login")

# Gemini model variants we will compare (all served via Vertex AI)
MODELS = [
    "google/gemini-2.5-pro",
    "google/gemini-2.5-flash",
    "google/gemini-1.5-pro-001",
    "google/gemini-1.5-flash-001",
]
print("\nModels to compare:")
for m in MODELS:
    print(f"  {m}")


In [ ]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [ ]:
messages

In [ ]:
# Generate the challenging question using Vertex AI
response = client.chat.completions.create(
    model="google/gemini-2.5-flash",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


In [ ]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Vertex AI model comparison

We will run the same question through four Gemini model variants, all served via Vertex AI:

| # | Model |
|---|-------|
| 1 | `google/gemini-2.5-pro` |
| 2 | `google/gemini-2.5-flash` |
| 3 | `google/gemini-1.5-pro-001` |
| 4 | `google/gemini-1.5-flash-001` |


In [ ]:
# Competitor 1 — Gemini 2.5 Pro
model_name = "google/gemini-2.5-pro"

response = client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


In [ ]:
# Competitor 2 — Gemini 2.5 Flash
model_name = "google/gemini-2.5-flash"

response = client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


In [ ]:
# Competitor 3 — Gemini 1.5 Pro (previous generation, still capable)
model_name = "google/gemini-1.5-pro-001"

response = client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


In [ ]:
# Competitor 4 — Gemini 1.5 Flash (fast & lightweight previous generation)
model_name = "google/gemini-1.5-flash-001"

response = client.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


In [ ]:
# So where are we?

print(competitors)
print(answers)


In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

In [ ]:
# Judgement time! Use Gemini 2.5 Pro as the judge (via Vertex AI)
response = client.chat.completions.create(
    model="google/gemini-2.5-pro",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


In [ ]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>